# 7주차 — LoRA와_QLoRA_비교


In [1]:
# 최초 실행 셀 · 설치 후 아래 import 셀 실행
# 같은 런타임에서 다른 버전의 transformers를 이미 import했다면 세션 재시작 후 처음부터 실행
%pip -q install "transformers==4.57.6" "peft==0.17.1" "accelerate==1.10.1" "numpy>=1.26,<2.3" "pandas>=2.2,<3" "matplotlib>=3.8,<3.11" "scikit-learn>=1.4,<1.8"
%pip -q install "ftfy>=6.2,<7" requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00


In [2]:
import os, sys, json, math, time, random, hashlib, gc, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = os.environ.get("LAB_SMOKE_TEST") == "1"  # 제작 검증용 · 수업에서는 False
torch.set_num_threads(min(4, os.cpu_count() or 1))
RESULT_DIR = Path("lab_results_week7")
RESULT_DIR.mkdir(exist_ok=True)
ENV = {"python":sys.version.split()[0], "torch":torch.__version__,
       "device":str(DEVICE), "gpu":torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else None,
       "seed":SEED, "smoke_test":SMOKE}
display(pd.DataFrame([ENV]))

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

def cleanup():
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

def save_json(name, obj):
    (RESULT_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def download_results(label):
    import shutil
    archive = shutil.make_archive(label, "zip", RESULT_DIR)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("저장 위치:", Path(archive).resolve())

STUDENT_ID=""  # 학번 입력
STUDENT_NAME=""  # 이름 입력
ANSWERS={}
PASSED=set()


,python,torch,device,gpu,seed,smoke_test
0,3.13.15,2.11.0+cu130,cuda,Tesla T4,42,False


In [3]:
MODEL_ID = "distilbert/distilgpt2"
REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def load_base():
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=REVISION, use_safetensors=True,
        attn_implementation="eager", torch_dtype=torch.float32)
    # 두 학습 방식의 dropout 조건 통일
    for module in model.modules():
        if isinstance(module, nn.Dropout): module.p = 0.0
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    return model.to(DEVICE)
print("모델:", MODEL_ID, "revision:", REVISION)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

모델: distilbert/distilgpt2 revision: 2290a62682d06624634c1f46a6ad5be0f47f38aa


In [4]:
# 수업용으로 직접 구성한 가상 지원 요청 · 실제 개인정보/외부 데이터 불포함
# 같은 원문 시나리오의 말투 변형은 반드시 같은 split에 배치
SCENARIOS = {
 "network":["The lab wifi disconnects", "The wired internet is unavailable", "The router loses its signal",
            "The network connection drops", "The campus wifi has no connection", "The ethernet port is offline",
            "The wireless connection keeps failing", "The internet connection is unstable"],
 "account":["My login password was rejected", "My account is locked", "I cannot sign into my account",
            "The password reset link expired", "My login credentials do not work", "Account authentication fails",
            "The sign in page rejects my password", "My account access is blocked"],
 "hardware":["The monitor has a broken screen", "The keyboard has broken keys", "The mouse button is broken",
             "The laptop battery is damaged", "The printer has a damaged tray", "The desktop fan is broken",
             "The display cable is physically damaged", "The computer power button is broken"],
 "software":["The spreadsheet application crashes", "The editor application will not start", "The browser application freezes",
             "The presentation program crashes", "The drawing application fails to launch", "The code editor freezes",
             "The calendar application crashes on launch", "The document application stops responding"]
}
WRAPPERS = ["{issue}. {urgency}", "Please help: {issue}. {urgency}", "Support request: {issue}. {urgency}"]
URGENCIES = [("This is urgent and blocks today's class.", "high"),
             ("This is routine and can wait until next week.", "normal")]

def make_records():
    rows=[]
    for category, issues in SCENARIOS.items():
        for i, issue in enumerate(issues):
            split = "train" if i<6 else ("validation" if i==6 else "test")
            for urgency, priority in URGENCIES:
                for j, wrapper in enumerate(WRAPPERS):
                    rows.append({"id":f"{category}-{i}-{priority}-{j}", "source_id":f"{category}-{i}",
                                 "split":split, "text":wrapper.format(issue=issue, urgency=urgency),
                                 "category":category, "priority":priority,
                                 "answer":f"category={category}; priority={priority}"})
    return pd.DataFrame(rows)

data = make_records()
train_df = data[data.split=="train"].reset_index(drop=True)
val_df = data[data.split=="validation"].reset_index(drop=True)
test_df = data[data.split=="test"].reset_index(drop=True)
for a,b in [(train_df,val_df),(train_df,test_df),(val_df,test_df)]:
    assert set(a.source_id).isdisjoint(b.source_id)
DATA_HASH = hashlib.sha256(data.to_json(orient="records").encode()).hexdigest()
print("분할:", data.groupby("split").size().to_dict(), "지문:", DATA_HASH[:12])
display(train_df.head(3))


분할: {'test': 24, 'train': 144, 'validation': 24} 지문: 387a20a315ef


,id,source_id,split,text,category,priority,answer
0,network-0-high-0,network-0,train,The lab wifi disconnects. This is urgent and b...,network,high,category=network; priority=high
1,network-0-high-1,network-0,train,Please help: The lab wifi disconnects. This is...,network,high,category=network; priority=high
2,network-0-high-2,network-0,train,Support request: The lab wifi disconnects. Thi...,network,high,category=network; priority=high


## 가중치를 동결하고 저랭크 업데이트만 학습할 수 있는가?


- 구현: 원 가중치 W 동결, LoRA forward와 학습 파라미터 수 계산
- 식: $y=xW^T+(\alpha/r)xA^TB^T$
- B=0 초기화에서 base와 출력이 같은지 확인
- 첫 backward에서 A와 B의 gradient가 다른 이유 생각해보기


In [5]:
class MyLoRALinear(nn.Module):
    def __init__(self,W,rank=2,alpha=4):
        super().__init__();self.scale=alpha/rank
        self.weight=nn.Parameter(W.clone(),requires_grad=False)
        self.A=nn.Parameter(torch.randn(rank,W.shape[1])*0.01)
        self.B=nn.Parameter(torch.zeros(W.shape[0],rank))
    def forward(self,x):
        return F.linear(x,self.weight)+(x@self.A.T@self.B.T)*self.scale
def trainable_count(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


In [6]:
torch.manual_seed(SEED);W=torch.randn(12,8);x=torch.randn(5,8);target=torch.randn(5,12)
toy=MyLoRALinear(W);initial=toy.weight.detach().clone()
assert not toy.weight.requires_grad and trainable_count(toy)==40
assert torch.allclose(toy(x),F.linear(x,W),atol=1e-6)
loss=F.mse_loss(toy(x),target);loss.backward()
grad_table=pd.DataFrame([{"parameter":n,"requires_grad":p.requires_grad,
                         "gradient_norm":None if p.grad is None else float(p.grad.norm())} for n,p in toy.named_parameters()])
display(grad_table)
assert toy.weight.grad is None and torch.all(toy.A.grad==0) and toy.B.grad.norm()>0
optimizer=torch.optim.SGD([p for p in toy.parameters() if p.requires_grad],lr=0.1);optimizer.step()
assert torch.equal(toy.weight,initial)
grad_table.to_csv(RESULT_DIR/"toy_lora_gradients.csv",index=False);PASSED.add("7-1_lora")
del toy,optimizer


,parameter,requires_grad,gradient_norm
0,weight,False,NaN
1,A,True,0.000000
2,B,True,0.025741


A의 gradient 는 B 값에 비례하는데, B=0 이므로 처음 A의 gradient는 0

A=0, B=0 인 경우 애초에 B는 input도 0이 들어오는 꼴이라, gradient 가 0이됨 => 둘 다 0으로 초기화하면 안되는 이유

## 전체 학습과 두 rank 중 무엇을 선택할 것인가?

- full SFT / LoRA rank 4 / LoRA rank 16 비교
- 같은 base·데이터·step·유효 batch·lr·평가 입력 유지
- LoRA alpha/r=2, dropout=0 고정
- 제공 설정은 동등 예산 비교임. 방법별 최적 hyperparameter 탐색 결과는 아님


In [7]:
MAX_LENGTH = 128
def make_prompt(text):
    return ("Classify the support request. Categories: network, account, hardware, software. "
            "Priorities: high, normal.\nRequest: " + text + "\nAnswer:")

def encode_row(row):
    prompt_ids = tokenizer.encode(make_prompt(row["text"]), add_special_tokens=False)
    answer_ids = tokenizer.encode(" " + row["answer"], add_special_tokens=False) + [tokenizer.eos_token_id]
    ids = prompt_ids + answer_ids
    assert len(ids)<=MAX_LENGTH, "길이 초과: MAX_LENGTH 또는 입력 확인 필요"
    # 정답 부분과 EOS에만 손실 적용 · prompt와 padding은 -100
    labels = [-100]*len(prompt_ids) + answer_ids
    return {"input_ids":ids, "labels":labels}

train_items = [encode_row(r) for r in train_df.to_dict("records")]
val_items = [encode_row(r) for r in val_df.to_dict("records")]

def collate(items):
    length=max(len(x["input_ids"]) for x in items)
    ids=[]; masks=[]; labels=[]
    for x in items:
        n=len(x["input_ids"]); pad=length-n
        ids.append(x["input_ids"]+[tokenizer.pad_token_id]*pad)
        masks.append([1]*n+[0]*pad)
        labels.append(x["labels"]+[-100]*pad)
    return {k:torch.tensor(v,device=DEVICE) for k,v in
            {"input_ids":ids,"attention_mask":masks,"labels":labels}.items()}

batch = collate(train_items[:2])
assert torch.all(batch["labels"][batch["attention_mask"]==0] == -100)
display(pd.DataFrame({"token":tokenizer.convert_ids_to_tokens(batch["input_ids"][0].tolist()),
                      "label":batch["labels"][0].tolist(),
                      "loss_target":(batch["labels"][0]!=-100).tolist()}))


,token,label,loss_target
0,Class,-100,False
1,ify,-100,False
2,Ġthe,-100,False
3,Ġsupport,-100,False
4,Ġrequest,-100,False
5,.,-100,False
6,ĠCategories,-100,False
7,:,-100,False
8,Ġnetwork,-100,False
9,",",-100,False


In [8]:
# 학습 설정 변경은 이 셀에서만 수행 · 6~7주차 비교 시 설정 일치 필요
STEPS = 3 if SMOKE else (60 if DEVICE.type=="cuda" else 8)
BATCH_SIZE = 2 if SMOKE else (4 if DEVICE.type=="cuda" else 1)
ACCUM_STEPS = 1 if SMOKE else 2
LEARNING_RATE = 1e-4
TRAIN_CONFIG = dict(steps=STEPS,batch_size=BATCH_SIZE,accum_steps=ACCUM_STEPS,
                    lr=LEARNING_RATE,max_length=MAX_LENGTH,seed=SEED,dropout=0.0)
if DEVICE.type=="cpu":
    print("CPU 축소 모드: 동작 확인용. 본 수업의 학습 효과 비교는 GPU 60-step 조건 권장")

@torch.no_grad()
def evaluate_loss(model, items):
    model.eval(); total_loss=0.0; count=0; correct=0
    for start in range(0,len(items),4):
        b=collate(items[start:start+4])
        out=model(**b)
        targets=b["labels"][:,1:]
        keep=targets!=-100
        n=int(keep.sum())
        total_loss+=float(out.loss)*n; count+=n
        preds=out.logits[:,:-1].argmax(-1)
        correct+=int(((preds==targets)&keep).sum())
    return {"answer_nll":total_loss/count, "teacher_forced_token_acc":correct/count}

@torch.no_grad()
def predict(model, text, max_new_tokens=18):
    model.eval()
    inputs=tokenizer(make_prompt(text),return_tensors="pt").to(DEVICE)
    output=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,use_cache=True,
                          pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0,inputs.input_ids.shape[1]:],skip_special_tokens=True).strip()

def evaluate_generation(model, frame, limit=None):
    rows=[]
    for r in frame.head(limit if limit is not None else len(frame)).to_dict("records"):
        pred=predict(model,r["text"])
        match=re.fullmatch(r"category=(network|account|hardware|software); priority=(high|normal)",pred)
        rows.append({"id":r["id"],"text":r["text"],"expected":r["answer"],"prediction":pred,
                     "exact_match":pred==r["answer"],"valid_format":bool(match)})
    table=pd.DataFrame(rows)
    return table, {"exact_match":float(table.exact_match.mean()),"format_rate":float(table.valid_format.mean())}

def train_run(model, label):
    set_seed(SEED)
    rng=np.random.default_rng(SEED)
    trainable=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(trainable,lr=LEARNING_RATE,weight_decay=0.0)
    # 메모리 측정: 해당 모델 단독 상주 + optimizer/activation 포함 최대 할당량
    cleanup(); sync()
    if DEVICE.type=="cuda": torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter(); history=[]
    model.train(); optimizer.zero_grad(set_to_none=True)
    for step in range(STEPS):
        loss_sum=0.0
        for _ in range(ACCUM_STEPS):
            indices=rng.integers(0,len(train_items),size=BATCH_SIZE)
            batch=collate([train_items[int(i)] for i in indices])
            loss=model(**batch).loss
            assert torch.isfinite(loss), "비정상 손실 발견"
            (loss/ACCUM_STEPS).backward()
            loss_sum+=float(loss.detach())/ACCUM_STEPS
        nn.utils.clip_grad_norm_(trainable,1.0)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
        history.append({"step":step+1,"loss":loss_sum})
        if step==0 or (step+1)%10==0 or step+1==STEPS:
            print(f"{label}: {step+1}/{STEPS}, loss={loss_sum:.4f}")
    sync(); seconds=time.perf_counter()-start
    stats={"method":label,"seconds":seconds,"trainable_parameters":sum(p.numel() for p in trainable),
           "total_parameters":sum(p.numel() for p in model.parameters()),
           "peak_allocated_MiB":torch.cuda.max_memory_allocated()/2**20 if DEVICE.type=="cuda" else None}
    del optimizer
    return pd.DataFrame(history),stats

def run_signature():
    return {"model_id":MODEL_ID,"revision":REVISION,"data_hash":DATA_HASH,
            "train_config":TRAIN_CONFIG,"environment":ENV}


In [9]:
generation_val=val_df.groupby(["category","priority"],sort=True).head(1).reset_index(drop=True)
USE_WEEK6_RESULT=False  # True: 6주차 v2 summary JSON 한 개 업로드
if USE_WEEK6_RESULT:
    from google.colab import files
    uploaded=files.upload(); names=[n for n in uploaded if n.endswith(".json")]
    assert len(names)==1
    full_record=json.loads(uploaded[names[0]].decode("utf-8"))
    assert full_record.get("version")==2,"제출형 v2 결과 파일 필요"
    current=run_signature()
    for key in ["model_id","revision","data_hash","train_config"]:
        assert full_record["signature"][key]==current[key],f"비교 조건 불일치: {key}"
    comparable=all(full_record["signature"]["environment"].get(k)==ENV.get(k) for k in ["device","gpu","torch"])
else:
    base=load_base()
    history,stats=train_run(base,"full_sft")
    metrics=evaluate_loss(base,val_items)
    predictions,gen=evaluate_generation(base,generation_val)
    full_record={"version":2,"signature":run_signature(),"stats":stats,"after":{**metrics,**gen}}
    history.to_csv(RESULT_DIR/"full_history.csv",index=False)
    del base;cleanup();comparable=True
display(pd.DataFrame([{**full_record["stats"],**full_record["after"]}]))


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


full_sft: 1/60, loss=5.3014
full_sft: 10/60, loss=0.0180
full_sft: 20/60, loss=0.0010
full_sft: 30/60, loss=0.0008
full_sft: 40/60, loss=0.0001
full_sft: 50/60, loss=0.0000
full_sft: 60/60, loss=0.0000


,method,seconds,trainable_parameters,total_parameters,peak_allocated_MiB,answer_nll,teacher_forced_token_acc,exact_match,format_rate
0,full_sft,10.09543,81912576,81912576,1723.981934,0.002021,1.0,1.0,1.0


In [10]:
from peft import LoraConfig,get_peft_model,PeftModel
RANKS=[4,16]; LORA_TARGETS=["c_attn","c_proj"]
lora_results=[];expected_logits={}
probe=collate(val_items[:1])
for rank in RANKS:
    base=load_base()
    config=LoraConfig(task_type="CAUSAL_LM",r=rank,lora_alpha=2*rank,lora_dropout=0.,
                      target_modules=LORA_TARGETS,fan_in_fan_out=True,bias="none")
    adapted=get_peft_model(base,config)
    names=[n for n,p in adapted.named_parameters() if p.requires_grad]
    assert names and all("lora_" in n for n in names)
    adapted.print_trainable_parameters()
    history,stats=train_run(adapted,f"lora_r{rank}")
    metrics=evaluate_loss(adapted,val_items)
    predictions,gen=evaluate_generation(adapted,generation_val)
    lora_results.append({**stats,**metrics,**gen,"rank":rank})
    adapted.eval()
    with torch.no_grad(): expected_logits[rank]=adapted(**probe).logits[0,-1].cpu().clone()
    adapted.save_pretrained(RESULT_DIR/f"adapter_rank{rank}",safe_serialization=True)
    tokenizer.save_pretrained(RESULT_DIR/f"adapter_rank{rank}")
    history.to_csv(RESULT_DIR/f"lora_rank{rank}_history.csv",index=False)
    predictions.to_csv(RESULT_DIR/f"lora_rank{rank}_validation.csv",index=False)
    del adapted,base;cleanup()
comparison=pd.DataFrame([{**full_record["stats"],**full_record["after"],"rank":None}]+lora_results)
comparison["trainable_percent"]=100*comparison.trainable_parameters/comparison.total_parameters
if not comparable:
    comparison.loc[comparison.method=="full_sft",["seconds","peak_allocated_MiB"]]=np.nan
display(comparison);comparison.to_csv(RESULT_DIR/"full_vs_lora.csv",index=False)
save_json("comparison_signature.json",run_signature())
PASSED.add("7-2_comparison")


trainable params: 202,752 || all params: 82,115,328 || trainable%: 0.2469
lora_r4: 1/60, loss=5.3014
lora_r4: 10/60, loss=4.8825
lora_r4: 20/60, loss=4.5569
lora_r4: 30/60, loss=3.8980
lora_r4: 40/60, loss=3.0012
lora_r4: 50/60, loss=2.0981
lora_r4: 60/60, loss=1.1424
trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804
lora_r16: 1/60, loss=5.3014
lora_r16: 10/60, loss=4.3589
lora_r16: 20/60, loss=2.9356
lora_r16: 30/60, loss=1.3154
lora_r16: 40/60, loss=0.4329
lora_r16: 50/60, loss=0.2521
lora_r16: 60/60, loss=0.1182


,method,seconds,trainable_parameters,total_parameters,peak_allocated_MiB,answer_nll,teacher_forced_token_acc,exact_match,format_rate,rank,trainable_percent
0,full_sft,10.095430,81912576,81912576,1723.981934,0.002021,1.000000,1.0,1.0,NaN,100.000000
1,lora_r4,5.112676,202752,82115328,591.430176,1.081047,0.777778,0.0,0.0,4.0,0.246911
2,lora_r16,4.471545,811008,82723584,600.895996,0.152774,0.939394,0.5,1.0,16.0,0.980383


### 추가 실험(60step 이상)을 통해 수렴한 성능 비교

## Adapter만 저장했는데 같은 모델을 복원할 수 있는가?


- 모델을 새로 로드하고 선택 adapter 결합
- 필요한 base ID·revision·adapter 경로 지정
- 저장 전후 동일 입력 logits 최대 오차 측정
- 선택 완료 후 test 24개 평가


In [12]:
BASE_ID=MODEL_ID
BASE_REVISION=REVISION
ADAPTER_PATH=str(RESULT_DIR/f"adapter_rank{SELECTED_RANK}")
assert BASE_ID==MODEL_ID and BASE_REVISION==REVISION
assert Path(ADAPTER_PATH)==RESULT_DIR/f"adapter_rank{SELECTED_RANK}"


In [13]:
reloaded=PeftModel.from_pretrained(load_base(),ADAPTER_PATH,is_trainable=False).eval()
with torch.no_grad(): restored=reloaded(**probe).logits[0,-1].cpu()
max_difference=float((restored-expected_logits[SELECTED_RANK]).abs().max())
assert torch.allclose(restored,expected_logits[SELECTED_RANK],atol=1e-4,rtol=1e-4)
inventory=[{"file":p.name,"bytes":p.stat().st_size} for p in Path(ADAPTER_PATH).iterdir() if p.is_file()]
display(pd.DataFrame(inventory));print("재로드 logits 최대 차이:",max_difference)
test_predictions,test_metrics=evaluate_generation(reloaded,test_df)
display(test_predictions);display(pd.DataFrame([test_metrics]))
test_predictions.to_csv(RESULT_DIR/"final_test.csv",index=False)
save_json("reload_and_test.json",{"max_difference":max_difference,"inventory":inventory,"test":test_metrics})
del reloaded;cleanup();PASSED.add("7-3_reload")


,file,bytes
0,adapter_config.json,854
1,merges.txt,456318
2,special_tokens_map.json,131
3,vocab.json,798156
4,adapter_model.safetensors,3248616
5,tokenizer.json,3557680
6,README.md,5196
7,tokenizer_config.json,507


재로드 logits 최대 차이: 0.0


,id,text,expected,prediction,exact_match,valid_format
0,network-7-high-0,The internet connection is unstable. This is u...,category=network; priority=high,category=software; priority=high,False,True
1,network-7-high-1,Please help: The internet connection is unstab...,category=network; priority=high,category=software; priority=high,False,True
2,network-7-high-2,Support request: The internet connection is un...,category=network; priority=high,category=software; priority=high,False,True
3,network-7-normal-0,The internet connection is unstable. This is r...,category=network; priority=normal,category=software; priority=normal,False,True
4,network-7-normal-1,Please help: The internet connection is unstab...,category=network; priority=normal,category=software; priority=normal,False,True
5,network-7-normal-2,Support request: The internet connection is un...,category=network; priority=normal,category=software; priority=normal,False,True
6,account-7-high-0,My account access is blocked. This is urgent a...,category=account; priority=high,category=hardware; priority=high,False,True
7,account-7-high-1,Please help: My account access is blocked. Thi...,category=account; priority=high,category=hardware; priority=high,False,True
8,account-7-high-2,Support request: My account access is blocked....,category=account; priority=high,category=account; priority=high,True,True
9,account-7-normal-0,My account access is blocked. This is routine ...,category=account; priority=normal,category=hardware; priority=normal,False,True


,exact_match,format_rate
0,0.333333,1.0


## LoRA를 쓰면 전체 메모리도 같은 비율로 줄어드는가?


- 구분: weight / gradient / optimizer state / activation / quantization metadata
- 구현: 파라미터 수와 bit 수로 이론 가중치 MiB 계산
- full/LoRA 실측은 앞의 표 사용 · CPU 메모리 빈칸을 0으로 해석하지 않음
- QLoRA 실측은 선택 GPU 셀 실행 시에만 기록


In [14]:
def ideal_weight_mib(parameter_count,bits):
    return parameter_count*bits/8/(2**20)


In [15]:
assert math.isclose(ideal_weight_mib(2**20,8),1.0)
params=full_record["stats"]["total_parameters"]
memory_table=pd.DataFrame([{"bits":bits,"ideal_weight_MiB":ideal_weight_mib(params,bits)} for bits in [32,16,8,4]])
display(memory_table);memory_table.to_csv(RESULT_DIR/"ideal_memory.csv",index=False)
save_json("qlora_status.json",{"status":"not_measured","reason":"선택 GPU 실험 미실행"})
PASSED.add("7-4_memory_formula")


,bits,ideal_weight_MiB
0,32,312.47168
1,16,156.23584
2,8,78.11792
3,4,39.05896
